In [1]:
import os
import glob

files = (
    glob.glob("/content/final_labels_SG2.xlsx", recursive=True)
    + glob.glob("/content/final_labels_SG1.xlsx", recursive=True)
)

if files:
    for file in files:
        print(file)
else:
    print("BABE Excel file not found.")
    print("Upload it using the Colab Files sidebar.")

BABE Excel file not found.
Upload it using the Colab Files sidebar.


In [2]:
import pandas as pd

file_path = next(
    (f for f in files if f.endswith("final_labels_SG2.xlsx")),
    None
)

if file_path is None:
    raise FileNotFoundError(
        "SG2 file not found. Check the printed paths or upload SG2."
    )

babe_df = pd.read_excel(file_path)

print("Dataset shape:", babe_df.shape)
print("\nColumn names:")
print(babe_df.columns.tolist())

display(babe_df.head())


FileNotFoundError: SG2 file not found. Check the printed paths or upload SG2.

In [ ]:
print("Dataset shape:", babe_df.shape)

print("\nMissing values:")
print(babe_df.isnull().sum())

print("\nData types:")
print(babe_df.dtypes)

print("\nUnique values per column:")
for col in babe_df.columns:
    print(f"\n{col}:")
    print(babe_df[col].value_counts(dropna=False).head(10))

In [ ]:
import pandas as pd

# Keep only rows with an agreed binary bias label
babe_clean = babe_df[
    babe_df["label_bias"].isin(["Biased", "Non-biased"])
].copy()

# Normalize text whitespace
babe_clean["text"] = (
    babe_clean["text"]
    .astype(str)
    .str.replace(r"\s+", " ", regex=True)
    .str.strip()
)

# Remove empty text
babe_clean = babe_clean[babe_clean["text"] != ""].copy()

# Audit exact duplicate sentences
print("Rows after removing No agreement labels:", len(babe_clean))
print("Exact duplicate texts:", babe_clean["text"].duplicated().sum())

# Report duplicates before deciding how to handle them
duplicate_rows = babe_clean[
    babe_clean["text"].duplicated(keep=False)
].sort_values("text")

display(duplicate_rows[["text", "news_link", "outlet", "label_bias"]].head(20))

print("\nBinary label distribution:")
print(babe_clean["label_bias"].value_counts())

print("\nLabel proportions:")
print(babe_clean["label_bias"].value_counts(normalize=True).round(3))

print("\nUnique article links:", babe_clean["news_link"].nunique())
print("Missing article links:", babe_clean["news_link"].isna().sum())

In [ ]:
# Work on a fresh copy
babe_work = babe_clean.copy()

# Count distinct labels assigned to each exact text
text_label_counts = (
    babe_work.groupby("text")["label_bias"]
    .nunique()
)

# Texts with conflicting annotations
conflicting_texts = text_label_counts[
    text_label_counts > 1
].index

print("Conflicting texts:", len(conflicting_texts))

conflict_audit = babe_work[
    babe_work["text"].isin(conflicting_texts)
][["text", "news_link", "outlet", "label_bias"]]

display(conflict_audit)

# Exclude conflicting text examples
babe_work = babe_work[
    ~babe_work["text"].isin(conflicting_texts)
].copy()

# Remove repeated copies of identical text
babe_work = babe_work.drop_duplicates(
    subset=["text"],
    keep="first"
).reset_index(drop=True)

print("\nRows after deduplication:", len(babe_work))
print("Remaining duplicate texts:", babe_work["text"].duplicated().sum())

print("\nFinal label distribution:")
print(babe_work["label_bias"].value_counts())

In [ ]:
# Missing links cannot be used as reliable article-group identifiers
linked_rows = babe_work[babe_work["news_link"].notna()].copy()

article_sizes = linked_rows.groupby("news_link").size()

print("Unique linked articles:", article_sizes.size)
print("Articles with multiple sentences:", (article_sizes > 1).sum())
print("Sentences from multi-sentence articles:", article_sizes[article_sizes > 1].sum())

print("\nSentences per article:")
print(article_sizes.describe())

print("\nLargest article groups:")
display(article_sizes.sort_values(ascending=False).head(10))

In [ ]:
# Export the cleaned BABE dataset
import pandas as pd
from google.colab import files

# Use the cleaned dataset created in your notebook
cleaned_babe = babe_work.copy().reset_index(drop=True)

# Save as CSV
output_file = "/content/babe_cleaned.csv"
cleaned_babe.to_csv(output_file, index=False, encoding="utf-8")

print("Cleaned dataset shape:", cleaned_babe.shape)
print("Saved to:", output_file)

# Download the CSV to your computer
files.download(output_file)

In [ ]:

from sklearn.model_selection import GroupShuffleSplit
import numpy as np
import pandas as pd

# Copy the deduplicated dataset
babe_model = babe_work.copy().reset_index(drop=True)

# Create article groups using URLs.
# Missing URLs get unique groups instead of being combined.
babe_model["article_group"] = babe_model["news_link"].astype("string")

missing_url = babe_model["article_group"].isna()
babe_model.loc[missing_url, "article_group"] = [
    f"missing_url_row_{i}"
    for i in babe_model.index[missing_url]
]

# Convert labels to numeric values
label_map = {
    "Non-biased": 0,
    "Biased": 1
}
babe_model["label"] = babe_model["label_bias"].map(label_map)

# Split by article group
splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=42
)

train_idx, test_idx = next(
    splitter.split(
        babe_model,
        y=babe_model["label"],
        groups=babe_model["article_group"]
    )
)

train_df = babe_model.iloc[train_idx].copy()
test_df = babe_model.iloc[test_idx].copy()

# Verify no article occurs in both sets
train_groups = set(train_df["article_group"])
test_groups = set(test_df["article_group"])

print("Training rows:", len(train_df))
print("Testing rows:", len(test_df))
print("Overlapping article groups:", len(train_groups & test_groups))

print("\nTraining label distribution:")
print(train_df["label_bias"].value_counts(normalize=True).round(3))

print("\nTesting label distribution:")
print(test_df["label_bias"].value_counts(normalize=True).round(3))

print("\nTraining publishers:", train_df["outlet"].nunique())
print("Testing publishers:", test_df["outlet"].nunique())


In [ ]:

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
    f1_score
)
import matplotlib.pyplot as plt

# Prepare input text and labels
X_train = train_df["text"].fillna("").astype(str)
X_test = test_df["text"].fillna("").astype(str)

y_train = train_df["label"].astype(int)
y_test = test_df["label"].astype(int)

# Convert text into TF-IDF features
tfidf = TfidfVectorizer(
    ngram_range=(1, 2),
    min_df=2,
    max_features=30000,
    sublinear_tf=True,
    stop_words="english"
)

X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf = tfidf.transform(X_test)

print("Training feature shape:", X_train_tfidf.shape)
print("Testing feature shape:", X_test_tfidf.shape)

# Train the classifier
lr_model = LogisticRegression(
    max_iter=1000,
    class_weight="balanced",
    random_state=42
)

lr_model.fit(X_train_tfidf, y_train)

# Make predictions
y_pred = lr_model.predict(X_test_tfidf)

# Evaluate
print("\nAccuracy:", round(accuracy_score(y_test, y_pred), 4))
print("Macro-F1:", round(f1_score(y_test, y_pred, average="macro"), 4))

print("\nClassification Report:")
print(
    classification_report(
        y_test,
        y_pred,
        labels=[0, 1],
        target_names=["Non-biased", "Biased"],
        digits=4,
        zero_division=0
    )
)

# Confusion matrix
cm = confusion_matrix(y_test, y_pred, labels=[0, 1])

display(
    ConfusionMatrixDisplay(
        confusion_matrix=cm,
        display_labels=["Non-biased", "Biased"]
    ).plot()
)

plt.title("BABE Baseline: TF-IDF + Logistic Regression")
plt.show()


In [ ]:

train_outlets = set(train_df["outlet"].dropna().unique())
test_outlets = set(test_df["outlet"].dropna().unique())

print("Publishers in both sets:", len(train_outlets & test_outlets))
print("Publishers only in training:", len(train_outlets - test_outlets))
print("Publishers only in testing:", len(test_outlets - train_outlets))

print("\nPublishers shared by both sets:")
print(sorted(train_outlets & test_outlets))

print("\nTest performance by publisher:")
publisher_results = test_df[["outlet", "label"]].copy()
publisher_results["prediction"] = y_pred

publisher_results["correct"] = (
    publisher_results["label"] == publisher_results["prediction"]
)

print(
    publisher_results.groupby("outlet")
    .agg(
        samples=("correct", "size"),
        accuracy=("correct", "mean")
    )
    .sort_values("samples", ascending=False)
    .round(3)
)


In [ ]:

from sklearn.model_selection import GroupShuffleSplit
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    classification_report
)
import pandas as pd

# Use only rows with a known publisher for this experiment
publisher_df = babe_model[
    babe_model["outlet"].notna()
].copy().reset_index(drop=True)

# Split by publisher, not by sentence or article
publisher_splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=0.25,
    random_state=42
)

train_idx, test_idx = next(
    publisher_splitter.split(
        publisher_df,
        y=publisher_df["label"],
        groups=publisher_df["outlet"]
    )
)

pub_train = publisher_df.iloc[train_idx].copy()
pub_test = publisher_df.iloc[test_idx].copy()

print("Training rows:", len(pub_train))
print("Testing rows:", len(pub_test))
print("Training publishers:", pub_train["outlet"].nunique())
print("Testing publishers:", pub_test["outlet"].nunique())
print(
    "Overlapping publishers:",
    len(set(pub_train["outlet"]) & set(pub_test["outlet"]))
)

print("\nHeld-out publishers:")
print(sorted(pub_test["outlet"].unique()))

print("\nTraining label distribution:")
print(pub_train["label_bias"].value_counts(normalize=True).round(3))

print("\nTesting label distribution:")
print(pub_test["label_bias"].value_counts(normalize=True).round(3))

# Train a fresh baseline
vectorizer_pub = TfidfVectorizer(
    ngram_range=(1, 2),
    min_df=2,
    max_features=30000,
    sublinear_tf=True,
    stop_words="english"
)

X_train_pub = vectorizer_pub.fit_transform(
    pub_train["text"].fillna("").astype(str)
)
X_test_pub = vectorizer_pub.transform(
    pub_test["text"].fillna("").astype(str)
)

model_pub = LogisticRegression(
    max_iter=1000,
    class_weight="balanced",
    random_state=42
)

model_pub.fit(X_train_pub, pub_train["label"])
pred_pub = model_pub.predict(X_test_pub)

print("\nPublisher-disjoint accuracy:",
      round(accuracy_score(pub_test["label"], pred_pub), 4))
print("Publisher-disjoint macro-F1:",
      round(f1_score(pub_test["label"], pred_pub, average="macro"), 4))

print("\nClassification report:")
print(classification_report(
    pub_test["label"],
    pred_pub,
    labels=[0, 1],
    target_names=["Non-biased", "Biased"],
    digits=4,
    zero_division=0
))



In [ ]:

import torch

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("Enable a GPU in Runtime > Change runtime type.")


In [ ]:

!pip install -q transformers datasets accelerate scikit-learn


In [ ]:

for name in ["babe_work", "babe_model", "babe_clean", "babe_df", "train_df", "test_df"]:
    print(f"{name}: {'available' if name in globals() else 'missing'}")


In [ ]:

import numpy as np
import torch

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding
)
from datasets import Dataset
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay
)
import matplotlib.pyplot as plt

MODEL_NAME = "FacebookAI/roberta-base"

# Verify the split is still the expected one
assert len(train_df) == 2961
assert len(test_df) == 708

# Map labels: 0 = Non-biased, 1 = Biased
train_data = train_df[["text", "label"]].copy()
test_data = test_df[["text", "label"]].copy()

train_data["text"] = train_data["text"].fillna("").astype(str)
test_data["text"] = test_data["text"].fillna("").astype(str)

train_data["label"] = train_data["label"].astype(int)
test_data["label"] = test_data["label"].astype(int)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

train_dataset = Dataset.from_pandas(
    train_data.reset_index(drop=True)
)
test_dataset = Dataset.from_pandas(
    test_data.reset_index(drop=True)
)

def tokenize_batch(batch):
    return tokenizer(
        batch["text"],
        truncation=True,
        max_length=256
    )

train_dataset = train_dataset.map(
    tokenize_batch,
    batched=True,
    remove_columns=["text"]
)
test_dataset = test_dataset.map(
    tokenize_batch,
    batched=True,
    remove_columns=["text"]
)

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

print("Training samples:", len(train_dataset))
print("Testing samples:", len(test_dataset))
print("GPU:", torch.cuda.get_device_name(0))


In [ ]:

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=2,
    id2label={0: "Non-biased", 1: "Biased"},
    label2id={"Non-biased": 0, "Biased": 1}
)

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)

    return {
        "accuracy": accuracy_score(labels, predictions),
        "macro_f1": f1_score(
            labels, predictions, average="macro"
        )
    }

training_args = TrainingArguments(
    output_dir="/content/newsbias_roberta",
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="eval_macro_f1",
    greater_is_better=True,
    fp16=True,
    logging_steps=50,
    report_to="none",
    seed=42,
    data_seed=42
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    data_collator=data_collator,
    compute_metrics=compute_metrics
)

trainer.train()


In [ ]:

pred_output = trainer.predict(test_dataset)

y_true = pred_output.label_ids
y_pred_roberta = np.argmax(pred_output.predictions, axis=-1)

print(
    "RoBERTa accuracy:",
    round(accuracy_score(y_true, y_pred_roberta), 4)
)
print(
    "RoBERTa macro-F1:",
    round(f1_score(y_true, y_pred_roberta, average="macro"), 4)
)

print("\nClassification report:")
print(classification_report(
    y_true,
    y_pred_roberta,
    labels=[0, 1],
    target_names=["Non-biased", "Biased"],
    digits=4,
    zero_division=0
))

ConfusionMatrixDisplay.from_predictions(
    y_true,
    y_pred_roberta,
    labels=[0, 1],
    display_labels=["Non-biased", "Biased"]
)
plt.title("BABE: RoBERTa Sentence Bias Detection")
plt.show()


In [ ]:

import pandas as pd
from sklearn.model_selection import GroupShuffleSplit

# Recreate the dataset if needed
if "babe_model" not in globals():
    if "babe_work" not in globals():
        raise NameError(
            "babe_model and babe_work are missing. "
            "Reload the cleaned BABE dataset first."
        )
    babe_model = babe_work.copy().reset_index(drop=True)

df = babe_model.copy().reset_index(drop=True)

# Ensure binary labels
if "label" not in df.columns:
    df["label"] = df["label_bias"].map({
        "Non-biased": 0,
        "Biased": 1
    })

df = df.dropna(subset=["text", "label"]).copy()
df["text"] = df["text"].astype(str)
df["label"] = df["label"].astype(int)

# Each article is a group. Missing URLs receive unique groups.
df["article_group"] = df["news_link"].astype("string")
missing = df["article_group"].isna()

df.loc[missing, "article_group"] = [
    f"missing_url_{i}" for i in df.index[missing]
]

# First: reserve 15% for the final test set
split1 = GroupShuffleSplit(
    n_splits=1, test_size=0.15, random_state=42
)

trainval_idx, test_idx = next(
    split1.split(df, groups=df["article_group"])
)

trainval_df = df.iloc[trainval_idx].copy()
test_df_final = df.iloc[test_idx].copy()

# Second: reserve approximately 15% of all data for validation
# 0.15 / 0.85 = 0.17647
split2 = GroupShuffleSplit(
    n_splits=1, test_size=0.17647, random_state=43
)

train_idx, val_idx = next(
    split2.split(
        trainval_df,
        groups=trainval_df["article_group"]
    )
)

train_df_final = trainval_df.iloc[train_idx].copy()
val_df_final = trainval_df.iloc[val_idx].copy()

# Check article-group leakage
train_groups = set(train_df_final["article_group"])
val_groups = set(val_df_final["article_group"])
test_groups = set(test_df_final["article_group"])

print("Training rows:", len(train_df_final))
print("Validation rows:", len(val_df_final))
print("Test rows:", len(test_df_final))

print("\nOverlapping article groups:")
print("Train vs validation:", len(train_groups & val_groups))
print("Train vs test:", len(train_groups & test_groups))
print("Validation vs test:", len(val_groups & test_groups))

for name, part in [
    ("Train", train_df_final),
    ("Validation", val_df_final),
    ("Test", test_df_final)
]:
    print(f"\n{name} label proportions:")
    print(part["label"].value_counts(normalize=True).round(3))

print("\nUnique publishers per split:")
for name, part in [
    ("Train", train_df_final),
    ("Validation", val_df_final),
    ("Test", test_df_final)
]:
    print(name, part["outlet"].nunique())


In [ ]:

import numpy as np
import torch
from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    DataCollatorWithPadding,
    TrainingArguments,
    Trainer,
    EarlyStoppingCallback
)
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay
)
import matplotlib.pyplot as plt

MODEL_NAME = "FacebookAI/roberta-base"

def make_hf_dataset(frame):
    result = frame[["text", "label"]].copy()
    result["text"] = result["text"].fillna("").astype(str)
    result["label"] = result["label"].astype(int)
    return Dataset.from_pandas(result.reset_index(drop=True))

train_hf = make_hf_dataset(train_df_final)
val_hf = make_hf_dataset(val_df_final)
test_hf = make_hf_dataset(test_df_final)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize_batch(batch):
    return tokenizer(
        batch["text"],
        truncation=True,
        max_length=256
    )

train_hf = train_hf.map(
    tokenize_batch, batched=True, remove_columns=["text"]
)
val_hf = val_hf.map(
    tokenize_batch, batched=True, remove_columns=["text"]
)
test_hf = test_hf.map(
    tokenize_batch, batched=True, remove_columns=["text"]
)

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

print("Train:", len(train_hf))
print("Validation:", len(val_hf))
print("Test:", len(test_hf))
print("GPU:", torch.cuda.get_device_name(0))


In [ ]:

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=2,
    id2label={0: "Non-biased", 1: "Biased"},
    label2id={"Non-biased": 0, "Biased": 1}
)

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)

    return {
        "accuracy": accuracy_score(labels, predictions),
        "macro_f1": f1_score(
            labels, predictions, average="macro"
        )
    }

training_args = TrainingArguments(
    output_dir="/content/newsbias_roberta_clean",
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="eval_macro_f1",
    greater_is_better=True,
    fp16=True,
    logging_steps=50,
    report_to="none",
    seed=42,
    data_seed=42,
    save_total_limit=2
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_hf,
    eval_dataset=val_hf,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    callbacks=[
        EarlyStoppingCallback(early_stopping_patience=1)
    ]
)

trainer.train()

print("\nBest checkpoint:", trainer.state.best_model_checkpoint)
print("Best validation macro-F1:", trainer.state.best_metric)


In [ ]:

test_output = trainer.predict(test_hf)

y_true_final = test_output.label_ids
y_pred_final = np.argmax(test_output.predictions, axis=-1)

print(
    "Final test accuracy:",
    round(accuracy_score(y_true_final, y_pred_final), 4)
)
print(
    "Final test macro-F1:",
    round(f1_score(
        y_true_final, y_pred_final, average="macro"
    ), 4)
)

print("\nClassification report:")
print(classification_report(
    y_true_final,
    y_pred_final,
    labels=[0, 1],
    target_names=["Non-biased", "Biased"],
    digits=4,
    zero_division=0
))

ConfusionMatrixDisplay.from_predictions(
    y_true_final,
    y_pred_final,
    labels=[0, 1],
    display_labels=["Non-biased", "Biased"]
)
plt.title("NewsBias — Final BABE Test Results")
plt.show()


In [ ]:

import os

MODEL_DIR = "/content/newsbias_roberta_best"

trainer.save_model(MODEL_DIR)
tokenizer.save_pretrained(MODEL_DIR)

print("Model saved to:", MODEL_DIR)


In [ ]:

import shutil

shutil.make_archive(
    "/content/newsbias_roberta_best",
    "zip",
    "/content/newsbias_roberta_best"
)

print("Created /content/newsbias_roberta_best.zip")
